# Download dos dados

In [1]:
import kagglehub

kagglehub.login()

In [3]:
# Download latest version
path = kagglehub.competition_download("dogs-vs-cats")

print("Path to competition files:", path)


100%|██████████| 812M/812M [00:42<00:00, 20.0MB/s] 

Extracting files...


Path to competition files: /home/mateus-band/.cache/kagglehub/competitions/dogs-vs-cats


In [4]:
!mv ~/.cache/kagglehub/competitions/dogs-vs-cats ./

# Regularização das imagens

In [1]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm


def resize_and_center_crop(image_path: Path, target_size: int = 224) -> None:
    with Image.open(image_path) as img:
        # Converter para RGB caso haja canais diferentes
        img = img.convert("RGB")
        width, height = img.size

        # Calcular novas dimensões mantendo proporção (menor lado = target_size)
        if width < height:
            new_width = target_size
            new_height = int(height * (target_size / width))
        else:
            new_height = target_size
            new_width = int(width * (target_size / height))

        img_resized = img.resize((new_width, new_height), Image.Resampling.BILINEAR)

        # Extrair quadrado central de target_size x target_size
        left = (new_width - target_size) // 2
        top = (new_height - target_size) // 2
        right = left + target_size
        bottom = top + target_size

        img_cropped = img_resized.crop((left, top, right, bottom))

        # Sobrescrever imagem original
        img_cropped.save(image_path)


def process_directory(directory_path: str | Path, target_size: int = 224) -> None:
    directory = Path(directory_path)
    image_extensions = {".jpg", ".jpeg", ".png"}
    files = [f for f in directory.iterdir() if f.suffix.lower() in image_extensions]

    print(f"Processando {len(files)} imagens em '{directory}'...")
    for file_path in tqdm(files):
        resize_and_center_crop(file_path, target_size=target_size)

In [3]:
# Pastas descompactadas em data/dogs-vs-cats
train_dir = "dogs-vs-cats/train"
test_dir = "dogs-vs-cats/test1"

process_directory(train_dir, target_size=224)
process_directory(test_dir, target_size=224)

Processando 25000 imagens em 'dogs-vs-cats/train'...


100%|██████████| 25000/25000 [00:52<00:00, 480.54it/s]


Processando 12500 imagens em 'dogs-vs-cats/test1'...


100%|██████████| 12500/12500 [00:32<00:00, 388.66it/s]
